In [ ]:
# Create Spark session with Hive support enabled
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName("SPARK SQL")
    .master("local[*]")
    .config("spark.executor.memory","512M")
    .enableHiveSupport()
    .getOrCreate()
)

In [ ]:
spark

In [ ]:
# Read the employee_rec dataset
_schema="first_name string,last_name string,job_title string,dob string,email string,phone string,salary string,department_id string"
emp=spark.read.format("csv").schema(_schema).option("header",True).load("employee_rec.csv")

In [ ]:
# read the department data
_schema="department_id string,department_name string,description string,city string,state string,country string"
dept=spark.read.format("csv").schema(_schema).option("header",True).load("department_data.csv")

In [ ]:
# Check Spark SQL catalog implementation (hive or in-memory)
spark.conf.get("spark.sql.catalogImplementation")


In [100]:
db=spark.sql("show databases")
db.show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [ ]:
# Show all tables in default database
spark.sql("show tables in default").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  default|      emp|      false|
|         |dept_view|       true|
|         | emp_view|       true|
+---------+---------+-----------+



In [102]:
# register dataframes are temp views
emp.createOrReplaceTempView("emp_view")

In [103]:
dept.createOrReplaceTempView("dept_view")

In [ ]:
#  SQL query: filter employees by department_id
df=spark.sql("""
select*from emp_view
where department_id =1
""")

In [105]:
df.show()

+-----------+---------+--------------------+----------+--------------------+--------------------+------+-------------+
| first_name|last_name|           job_title|       dob|               email|               phone|salary|department_id|
+-----------+---------+--------------------+----------+--------------------+--------------------+------+-------------+
|       John|   Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506|            1|
|    Rachael|Rodriguez|         Media buyer|1966-12-02|griffinmary@examp...| +1-791-344-7586x548|544732|            1|
|Christopher| Callahan| Exhibition designer|1966-10-23| qwalter@example.com|001-947-745-3939x...|251057|            1|
|    Lindsey|   Huerta|Embryologist, cli...|1964-10-20|  psmith@example.net|   527.934.6665x1378|878257|            1|
|      David|   Harris|   Company secretary|1990-04-13|     nli@example.com|001-959-766-1180x...|249553|            1|
|      Brian|Hernandez|     Theatre manager|1978

In [106]:
df=spark.sql("""
select *,date_format(dob,'yyyy') as dob_year 
from emp_view
""")

In [107]:
df.show()

+----------+----------+--------------------+----------+--------------------+--------------------+------+-------------+--------+
|first_name| last_name|           job_title|       dob|               email|               phone|salary|department_id|dob_year|
+----------+----------+--------------------+----------+--------------------+--------------------+------+-------------+--------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653|            8|    1973|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836|            7|    1974|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900|           10|    1990|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506|            1|    1968|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)900-5337|604738|

In [ ]:
# Save DataFrame as a Hive table in Parquet format
df.write.format("parquet").saveAsTable("emp")

In [109]:
spark.sql("select*from emp").show()

+----------+---------+--------------------+----------+--------------------+--------------------+------+-------------+--------+
|first_name|last_name|           job_title|       dob|               email|               phone|salary|department_id|dob_year|
+----------+---------+--------------------+----------+--------------------+--------------------+------+-------------+--------+
|    Teresa|    Scott|        Youth worker|1992-10-02|kingdebbie@exampl...|          7428090040|575636|            6|    1992|
|    Steven|  Johnson|              Writer|2000-01-04|trujilloaaron@exa...|    734-476-6520x067|776495|            2|    2000|
|   Patrick|    Marsh|Special education...|2000-02-12|amyhancock@exampl...|  (932)890-6038x7394|539686|            2|    2000|
|   Anthony|   Thomas|              Writer|1978-11-06| tknight@example.org|     +1-499-363-2947|394798|           10|    1978|
|      Karl|     Kent|          Geochemist|1966-03-16|yjohnson@example.org|       (375)285-4892|717373|        

In [110]:
# show details of metadata
spark.sql("describe extended emp").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|          first_name|              string|   null|
|           last_name|              string|   null|
|           job_title|              string|   null|
|                 dob|              string|   null|
|               email|              string|   null|
|               phone|              string|   null|
|              salary|              string|   null|
|       department_id|              string|   null|
|            dob_year|              string|   null|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|            Database|             default|       |
|               Table|                 emp|       |
|        Created Time|Tue Mar 03 05:53:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 3.3.0|       |
|           

In [ ]:
# Drop table
spark.sql("drop table emp")

DataFrame[]